# モジュール1 — RAGの基礎
### ゼロから作る「検索して答える」しくみ

## このモジュールで学ぶこと

AI(大規模言語モデル、LLM)は、とても賢く見えますが、実は2つの弱点があります。

1. **学習していないことは知らない** — 学習が終わった後に起きたことや、会社の内部資料のような
   非公開の情報は知りません。
2. **知らないことでも、それらしく答えてしまうことがある**(これを「ハルシネーション」=幻覚と
   呼びます)。

この弱点を補う代表的な方法が **RAG(Retrieval-Augmented Generation / 検索拡張生成)** です。
考え方はとてもシンプルで、人間が「知らないことは調べてから答える」のと同じです。

> 💡 **たとえ話:** 新入社員に「うちの会社の返品ポリシーは?」と聞いたとき、優秀な新入社員は
> 記憶だけで適当に答えるのではなく、まず**マニュアルを調べて**から答えますよね。RAGはAIに
> この「調べる」という行動をさせる仕組みです。

**シナリオ:** あなたは架空のドローン会社「ニンバス・ロボティクス」のサポート担当です。
お客様サポート担当が同じような製品の質問に何度も答えています。この作業をAIに手伝わせるための
「検索」の部分を、このモジュールで自分の手で作ります。

**作るものの全体像(4つのステップ):**
1. **チャンク分割** — 長い文書を検索しやすい小さな単位に分ける
2. **埋め込み(エンベディング)** — 文章を「意味を表す数字の列(ベクトル)」に変換する
3. **索引(インデックス)作成** — ベクトルを高速に検索できるように整理する
4. **検索(リトリーブ)** — 質問に近い文章を見つけ出す

> 💡 **オフラインで動く設計:** このノートブックは最初、TF-IDFという軽量な方法を使います。
> インターネット接続やモデルのダウンロードが不要で、即座に動きます。教室で20人以上が同時に
> 実行しても問題ありません。授業の最後の「ボーナス」セクションで、本格的なニューラル埋め込み
> モデルに切り替える方法を学びます。


## 環境セットアップ(SageMaker ノートブックインスタンス)

- **インスタンスタイプ:** `ml.t3.medium` で十分です。モジュール3のボーナスセルまで試す予定
  がある場合は `ml.t3.large` を選ぶと余裕があります。
- **カーネル:** `conda_pytorch_p310`
- **初回のみ:** 下のセルで、このカーネルに入っていないパッケージをインストールします。
  すでにインストール済みの環境で再実行しても問題はありません(すぐに終わります)。

In [ ]:
# このノートブックインスタンスで一度だけ実行してください
# conda_pytorch_p310 には numpy は入っていますが、以下は入っていません:
%pip install --quiet faiss-cpu scikit-learn

In [ ]:
# セットアップ — 最初に実行してください
#
# numpy: 数値計算のための定番ライブラリ。ベクトル(数字の並び)を効率よく扱うために使います。
# faiss: Meta(旧Facebook)が開発した、大量のベクトルの中から「似ているもの」を
#        高速に探すためのライブラリ。今日はステップ4「索引作成」で使います。
# TfidfVectorizer: 文章を「単語(今回は文字)の出現頻度」をもとにベクトル化するための
#        ツール。scikit-learn(機械学習の定番ライブラリ)に含まれています。
import numpy as np
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer

def TODO(hint=""):
    """演習の未完成部分を示す関数です。TODO(...) の呼び出しを自分のコードに置き換えて
    ください。置き換えるまではエラーが出続けます(これは正常な動作です)。"""
    raise NotImplementedError(f"ここを実装してください。ヒント: {hint}")

print("準備完了です。")

## ステップ1: 元になる文書

ニンバス・ロボティクスのサポート文書を5つ用意しました。実際のシステムでは、これらは
ナレッジベースやPDFマニュアル、問い合わせ履歴などから来ることが多いですが、この演習では
RAGの仕組みに集中するため、単純な文字列として扱います。

In [ ]:
# DOCS = ニンバス・ロボティクスのサポート文書(今日のRAGが検索する対象)。
# 実際のシステムではデータベースやPDFから読み込みますが、今日はコードに直接書いています。
DOCS = [
    "ニンバス・スカウトドローンの飛行時間は20分で、価格は249ドルです。",
    "ニンバス・カーゴドローンは最大5キログラムまで運搬でき、価格は899ドルです。",
    "返品は購入から30日以内で、元の梱包があれば受け付けます。",
    "ニンバス・ロボティクスの保証は、製造上の欠陥について12か月間保証します。",
    "ニンバスのドローンのバッテリーをフル充電するには約90分かかります。",
]
for d in DOCS:
    print("-", d)

## ステップ2: チャンク分割(チャンキング)とは何か、なぜ必要か

**なぜ文書をそのまま使わないのか?**

長い文書を1つのまとまりとして扱うと、複数の話題が混ざってしまい、「どの部分が質問の答えに
なっているか」がぼやけてしまいます。文書を小さな単位(チャンク)に分けておくことで、検索が
質問に対して**ピンポイントで**関連する部分を指し示せるようになります。

**日本語ならではの注意点:** 英語は単語の間にスペースがありますが、日本語にはありません。
そのため、このノートブックでは「単語数」ではなく **「文字数」** を基準にチャンクを分割します。

**あなたの番:** `chunk_documents` を完成させてください。各文書を、最大 `max_chars` 文字ずつの
かたまりに分割する関数です。

*(わからなければ、すぐ下の「レスキューセル」を実行してかまいません。それも学習の一部です。)*

In [ ]:
def chunk_documents(docs, max_chars=60):
    chunks = []
    for doc in docs:
        # doc は1つの文書(例:「ニンバス・スカウトドローンの飛行時間は...」)。
        # これを max_chars 文字ずつの小さな塊(チャンク)に分割し、chunks に追加します。
        # TODO: doc を max_chars 文字ずつに区切り、chunks に追加してください
        # ヒント: 文字列のスライス doc[i:i+max_chars] を使います
        TODO("range(0, len(doc), max_chars) でスライスして chunks に append する")
    return chunks

chunks = chunk_documents(DOCS)
print(f"{len(chunks)} 個のチャンクが作成されました")
for c in chunks:
    print("-", c)

In [ ]:
# --- レスキューセル: 先に動くコードを見たい場合はこちらを実行してください ---
def chunk_documents(docs, max_chars=60):
    chunks = []
    for doc in docs:
        # range(0, len(doc), max_chars) は「0, 60, 120, ...」のように
        # max_chars ずつ増える数字の列を作ります。これを doc の開始位置として使い、
        # doc[i:i+max_chars] で「i文字目からmax_chars文字分」を切り出します。
        for i in range(0, len(doc), max_chars):
            chunks.append(doc[i:i+max_chars])
    return chunks

chunks = chunk_documents(DOCS)
print(f"{len(chunks)} 個のチャンクが作成されました")
for c in chunks:
    print("-", c)

**気づいたこと:** 今回のサンプル文書はもともと短いため、チャンク数は元の文書数と
ほとんど変わりません。実際の長いマニュアルやFAQページでは、1つの文書が何十個ものチャンクに
分かれることになります。

## ステップ3: 埋め込み(エンベディング)— 文章を数字に変える

**埋め込みとは何か、直感的に理解する**

埋め込みとは、文章の「意味」を表す数字の並び(ベクトル)のことです。地図上の座標を
イメージしてください — 近い場所にある建物は座標も近い値になりますよね。埋め込みも同じで、
**似た意味を持つ文章は、似たベクトル(近い座標)になる**ように作られています。

**今回使う方法:TF-IDF**

このステップでは、単語(今回は文字の並び)の出現頻度をもとにベクトルを作る「TF-IDF」という
軽量な手法を使います。100%ローカルで動き、即座に計算できます。

**日本語ならではの注意点:** 日本語には単語の区切り(スペース)がないため、「単語」単位では
なく **文字2〜3文字のかたまり(文字N-gram)** を単位にして頻度を数えます。たとえば
「バッテリー」という単語は「バッ」「ッテ」「テリ」「リー」のような細かいかたまりに分解されて
数えられます。

> ⚠️ **この方法の限界(あとで確認します):** 文字の並びが似ているかどうかだけを見ているので、
> 「本当の意味」を理解しているわけではありません。たとえば「バッテリー」と「電池」は人間には
> 同じ意味だとわかりますが、文字はまったく重なっていないため、TF-IDFはこの2つを「関係ない
> 言葉」として扱ってしまいます。これをチェックポイント演習で実際に確認します。

**あなたの番:** `embed_chunks_local` を完成させてください。`TfidfVectorizer` を `chunks` に
対して学習・変換し、`float32` 型のNumPy配列として返してください(FAISSは `float32` を
必要とします)。

In [ ]:
def embed_chunks_local(chunks):
    # analyzer="char_wb", ngram_range=(2,3)
    #   → 日本語には英語のようなスペース区切りがないため、「単語」ではなく
    #      「文字2〜3文字の連続したかたまり」を数える単位として使います。
    #      (例:「バッテリー」→「バッ」「ッテ」「テリ」「リー」)
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
    # TODO: vectorizer.fit_transform(chunks) の結果を .toarray() で密行列にし、
    # .astype("float32") でFAISS用の型に変換してください
    # ヒント:
    #   fit_transform(chunks) → chunks の中の文字の出現パターンを学習し、ベクトルに変換
    #   .toarray()            → 内部形式(疎行列)を、扱いやすい通常の配列に変換
    #   .astype("float32")    → FAISSはfloat32型のデータしか受け付けないため型変換
    vectors = TODO("vectorizer.fit_transform(chunks).toarray().astype('float32')")
    return vectors, vectorizer

vectors, vectorizer = embed_chunks_local(chunks)
print("ベクトルの形:", vectors.shape)  # (チャンク数, 語彙のサイズ) が表示されるはずです

In [ ]:
# --- レスキューセル ---
def embed_chunks_local(chunks):
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
    # fit_transform: chunks 全体から「どんな文字の並びがあるか(語彙)」を学習しつつ、
    #                各チャンクをその語彙に基づいたベクトルに変換する、という2つの処理を
    #                同時に行います(fit=学習 + transform=変換)。
    vectors = vectorizer.fit_transform(chunks).toarray().astype("float32")
    return vectors, vectorizer

vectors, vectorizer = embed_chunks_local(chunks)
print("ベクトルの形:", vectors.shape)

## ステップ4: FAISSによる索引(インデックス)作成

FAISSは、大量のベクトルの中から「似ているもの」を高速に探すためのライブラリです。図書館の
蔵書検索カタログのように、1件ずつ全部を見比べる代わりに、あらかじめ整理しておくことで検索を
速くします。

今回使う `IndexFlatL2` はFAISSの中でも一番シンプルな種類で、すべてのベクトルと正確に
距離を計算する「完全一致検索」です。文書数が少ない教材用としてはこれで十分です。

このステップは完成済みのコードです。空欄はありません — ここでは「FAISSが何をしているか」を
理解することが目的で、自分で実装することが目的ではありません。

In [ ]:
def build_index(vectors):
    # IndexFlatL2: FAISSの中で最もシンプルな索引の種類。
    #   「Flat」= 近似ではなく、すべてのベクトルと正確に距離計算する方式
    #   「L2」  = 距離の測り方の名前(ユークリッド距離。中学校で習う「直線距離」と同じ考え方)
    # vectors.shape[1] は1つのベクトルの次元数(長さ)を表します。
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(vectors)  # 作った索引に、すべてのチャンクのベクトルを登録します
    return index

index = build_index(vectors)
print(index.ntotal, "件のベクトルを索引に登録しました")

## ステップ5: 検索(リトリーブ)

ここまでの準備が実を結ぶステップです。質問文を同じ方法でベクトル化し、索引に対して
「一番近いチャンクは?」と問い合わせます。

**あなたの番:** `retrieve` を完成させてください。`vectorizer` を使って質問文(query)を
同じ方法でベクトル化し、索引を検索して上位 `k` 件のチャンク文字列を返してください。

In [ ]:
def retrieve(query, vectorizer, index, chunks, k=2):
    # 質問文(query)も、チャンクと「まったく同じ方法」でベクトル化する必要があります。
    # (学習済みの vectorizer を使うことで、同じ文字の数え方が保証されます)
    qvec = vectorizer.transform([query]).toarray().astype("float32")
    # TODO: index.search(qvec, k) で距離(distances)と位置(indices)を取得し、
    # indices に対応する chunks をリストとして返してください
    # ヒント:
    #   index.search(qvec, k) は (distances, indices) の組を返します。
    #   distances[0] = 質問に近い順の「距離」のリスト(小さいほど近い)
    #   indices[0]   = 質問に近い順の「chunksの中での位置番号」のリスト
    TODO("index.search(qvec, k) の戻り値 indices[0] を使って chunks から取り出す")

results = retrieve("充電にはどれくらい時間がかかりますか", vectorizer, index, chunks)
for r in results:
    print("-", r)

In [ ]:
# --- レスキューセル ---
def retrieve(query, vectorizer, index, chunks, k=2):
    qvec = vectorizer.transform([query]).toarray().astype("float32")
    distances, indices = index.search(qvec, k)
    # indices[0] は「1件目の質問に対する、近い順のチャンク番号リスト」です
    # (今回は質問を1件ずつしか渡していないため、常に [0] を使います)
    return [chunks[i] for i in indices[0]]

results = retrieve("充電にはどれくらい時間がかかりますか", vectorizer, index, chunks)
for r in results:
    print("-", r)

## チェックポイント演習: TF-IDFの限界を自分の目で見る

同じ「充電時間」についての質問を、言い方を変えて2通り試してみます。

1. `"充電にはどれくらい時間がかかりますか"` — 文書中の「充電」という文字をそのまま含む質問
2. `"電池が満タンになるまでの時間は"` — 意味は同じですが、「充電」ではなく「電池」「満タン」
   という**別の文字**を使った質問(人間なら同じ意味だとわかります)

2つの結果を見比べて、何が起きているか観察してください。

In [ ]:
print("Q1(「充電」を含む質問):")
for r in retrieve("充電にはどれくらい時間がかかりますか", vectorizer, index, chunks):
    print("  -", r)

print()
print("Q2(同じ意味だが「充電」という文字を含まない質問):")
for r in retrieve("電池が満タンになるまでの時間は", vectorizer, index, chunks):
    print("  -", r)

**観察できたこと:** Q1は正しくバッテリーの文書を見つけられますが、Q2は見つけられず、
まったく関係ない文書が上位に出てきます。TF-IDF(文字の重なり)は「文字」を見ているだけで、
「意味」を理解していないことがよくわかる例です。

この限界をどう解決するかは、ボーナスセクションで確認します。

## ボーナス(任意・インターネット接続と数分が必要): 本物のニューラル埋め込みモデルを使う

TF-IDFの代わりに、実際に「意味」を学習したニューラル埋め込みモデルを使うと、Q2のような
言い換えにも対応できるようになります。

> ⚠️ **重要な注意:** 下のセルでモデルをインストールすると、`numpy` や関連ライブラリの
> バージョンが変わることがあります。その結果、**Jupyterがカーネルの再起動を求める**ことが
> あります。再起動した場合、これまで実行したセルの内容(`chunks`、`vectors`、`index` などの
> 変数)はすべて失われるため、**「Restart Kernel and Run All Cells」を使って最初から
> 再実行してください**。これはノートブックの不具合ではなく、新しい重いライブラリを追加した
> ときによく起こる正常な動作です。

このセルは演習の完了に**必須ではありません**。

## モデルの読み込み方法(2通り)

インターネット接続の状況によっては、Hugging Faceへの接続が不安定になることがあります
(実際にこのモジュールをテストした際に、SageMakerノートブックインスタンスのネットワークが
一時的にDNS解決できなくなり、Hugging Faceからのダウンロードに失敗したことがありました —
インスタンスの再起動で解決しましたが、授業中に同じことが起きると対応に時間を取られます)。

そこで、下のセルは **2通りの読み込み方法** に対応しています:

- `MODEL_SOURCE = "huggingface"`(デフォルト): Hugging Face Hubから直接ダウンロードします。
  インターネット接続が安定していれば、これが一番簡単です。
- `MODEL_SOURCE = "local"`: 事前にダウンロードしてS3などに置いてあるモデルを、ローカルの
  フォルダから読み込みます。インターネット接続に依存しないため、教室のネットワークが
  不安定な場合や、Hugging Faceへの接続が制限されている環境でも安定して動作します。

**このワークショップでは、S3バケット `llm-workshop-files` にモデルを事前配置して
あります。デフォルトで `MODEL_SOURCE = "local"` に設定済みなので、下のセルをそのまま
順番に実行するだけで大丈夫です。**

(ネットワーク環境によってはHugging Face経由を試したい場合のみ、`MODEL_SOURCE` を
`"huggingface"` に変更してください。)

In [ ]:
# 一度だけ実行してください(モデルは約120MB、初回は数分かかります)
%pip install --quiet sentence-transformers

In [ ]:
# このワークショップ用に、モデル一式をS3バケット(公開読み取り可能)に事前配置しています。
# --no-sign-request を付けることで、AWS認証情報なしで公開バケットから取得できます。
!aws s3 sync s3://llm-workshop-files/paraphrase-multilingual-MiniLM-L12-v2 ./models/paraphrase-multilingual-MiniLM-L12-v2 --no-sign-request

In [ ]:
# --- モデルの読み込み設定 ---
# "local":       上のセルでS3から取得したモデルをローカルフォルダから読み込む(デフォルト)
# "huggingface": Hugging Face Hubから直接ダウンロード(インターネット接続が必要)
MODEL_SOURCE = "local"  # ネットワークの都合でHugging Face経由を試したい場合は "huggingface" に変更
LOCAL_MODEL_PATH = "./models/paraphrase-multilingual-MiniLM-L12-v2"

# SentenceTransformer: 文章を「意味」を捉えたベクトルに変換する、本格的なAIモデルを
# 簡単に使うためのライブラリです。TF-IDFが「文字の並び」しか見ないのに対し、こちらは
# 大量の文章を学習することで、言葉の「意味」を理解したベクトルを作れます。
from sentence_transformers import SentenceTransformer

if MODEL_SOURCE == "local":
    # 1つ上のセルで s3://llm-workshop-files からダウンロード済みのフォルダを読み込みます。
    # Hugging Faceへの接続は一切発生しないため、教室のネットワーク状況に左右されません。
    model = SentenceTransformer(LOCAL_MODEL_PATH)
else:
    # 日本語を含む50以上の言語に対応した多言語モデル
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def embed_chunks_neural(chunks):
    # model.encode(...) が「文章を渡すと意味ベクトルを返す」処理をすべて行ってくれます。
    # 中身は複雑なニューラルネットワークですが、使う側はこの1行で済みます。
    return model.encode(chunks).astype("float32")

# TF-IDF用の index とは別に、ニューラル埋め込み専用の索引を新しく作ります
# (ベクトルの中身も次元数もTF-IDFとはまったく別物のため、混ぜて使うことはできません)
neural_vectors = embed_chunks_neural(chunks)
neural_index = faiss.IndexFlatL2(neural_vectors.shape[1])
neural_index.add(neural_vectors)

def retrieve_neural(query, k=2):
    qvec = model.encode([query]).astype("float32")
    distances, indices = neural_index.search(qvec, k)
    return [chunks[i] for i in indices[0]]

print(f"ニューラル埋め込みモデルの準備ができました。(読み込み元: {MODEL_SOURCE})")

### TF-IDF vs ニューラル埋め込み: 同じ質問で比較する

チェックポイント演習で失敗したQ2の質問を、両方の方法で試して比較してみましょう。

In [ ]:
query = "電池が満タンになるまでの時間は"

print("【TF-IDF(文字の重なりだけを見る)】")
for r in retrieve(query, vectorizer, index, chunks):
    print("  -", r)

print()
print("【ニューラル埋め込み(意味を理解する)】")
for r in retrieve_neural(query):
    print("  -", r)

**期待される結果:** TF-IDFは相変わらず的外れな文書を返しますが、ニューラル埋め込みは
「電池」と「バッテリー」、「満タン」と「充電」が意味的に近いことを理解し、正しい文書を
上位に返すはずです。これがTF-IDFとニューラル埋め込みの実力の差です。

## ボーナス+: 自分で質問を入力できるUI

コードを書き換えずに、いろいろな質問を試せるように、簡単な入力フォームを作ってみましょう。
`ipywidgets` というライブラリを使います。

**このUIでは、結果と一緒に「距離」という数字も表示します。** これは質問のベクトルと
チャンクのベクトルがどれくらい離れているかを表す数値で、**小さいほど「近い」= 似ている**
という意味です。

> ⚠️ **注意:** TF-IDFの距離とニューラル埋め込みの距離は、計算方法もスケールもまったく
> 違うので、**2つの数字を直接比べることはできません**(例:TF-IDFの1.2とニューラルの1.2が
> 同じ「近さ」を意味するわけではありません)。比較できるのは、**同じ方法の中で**「1位の
> 結果と2位の結果、どちらがより自信を持った答えか」という点だけです。

**1位と2位の距離の差に注目してください:**
- 差が大きい → 1位は自信を持った、はっきりした一致
- 差がほとんどない → 2位は「他に候補がないので仕方なく選んだ」弱い一致である可能性が高い

(今日のサンプル文書はたった5件しかないため、質問によっては本当に関連する文書が1件しか
なくても、2位が無理やり返されることがあります。これは実際のRAGシステムでもよくある
現象で、距離を見ることで「この結果は本当に信頼できるか」を判断する手がかりになります。)

In [ ]:
# ipywidgets: Jupyterノートブックの中に、ボタンや入力欄のような
# 簡単な操作画面(UI)を作るためのライブラリです。
import ipywidgets as widgets
from IPython.display import display, clear_output

# 質問を入力するテキストボックス
query_box = widgets.Text(
    value="",
    placeholder="ここに質問を日本語で入力してください",
    description="質問:",
    layout=widgets.Layout(width="500px"),
)
# 押すと検索を実行するボタン
search_button = widgets.Button(description="検索する", button_style="primary")
# 検索結果を表示するための領域
output_area = widgets.Output()

def retrieve_with_score(query, vectorizer, index, chunks, k=2):
    # retrieve() とほぼ同じ処理ですが、distances(距離)も一緒に返します。
    # distances は「質問のベクトルと、そのチャンクのベクトルがどれくらい離れているか」を
    # 表す数字で、小さいほど「近い」= 似ていることを意味します。
    qvec = vectorizer.transform([query]).toarray().astype("float32")
    distances, indices = index.search(qvec, k)
    return list(zip([chunks[i] for i in indices[0]], distances[0]))

def retrieve_neural_with_score(query, k=2):
    qvec = model.encode([query]).astype("float32")
    distances, indices = neural_index.search(qvec, k)
    return list(zip([chunks[i] for i in indices[0]], distances[0]))

def on_search_clicked(b):
    # ボタンが押されるたびに呼び出される関数です(b はボタン自身の情報ですが今回は使いません)
    with output_area:
        clear_output()  # 前回の結果を消してから新しい結果を表示します
        q = query_box.value
        if not q.strip():
            print("質問を入力してください。")
            return
        print(f"質問: {q}\n")
        print("【TF-IDF】")
        for rank, (chunk, dist) in enumerate(retrieve_with_score(q, vectorizer, index, chunks, k=2), start=1):
            print(f"  {rank}位 (距離: {dist:.3f})  {chunk}")
        print()
        print("【ニューラル埋め込み】")
        for rank, (chunk, dist) in enumerate(retrieve_neural_with_score(q, k=2), start=1):
            print(f"  {rank}位 (距離: {dist:.3f})  {chunk}")

# ボタンが押されたときに on_search_clicked を実行するように設定します
search_button.on_click(on_search_clicked)
# 3つの部品(入力欄・ボタン・出力領域)をノートブック上に表示します
display(query_box, search_button, output_area)

## まとめ

このモジュールでは、RAGの「検索」部分をゼロから作りました: チャンク分割 → 埋め込み →
索引作成 → 検索。そして、単純な文字の重なりだけを見る方法(TF-IDF)と、本当に意味を理解する
方法(ニューラル埋め込み)の違いを、実際に目で見て確認しました。

モジュール2では、ニンバス・ロボティクスの**在庫データ**(文書のような静的な情報ではなく、
今この瞬間の情報)にアクセスするための「ツール」を、MCPという仕組みを使って作ります。